# ClassForge ML: Fine-Tuning QLoRA (1-3B) para Lenguaje Natural → Comandos UML

Este notebook permite entrenar el modelo **ClassForge-UML** usando **QLoRA (4-bit)** sobre la GPU gratuita de Google Colab o Kaggle (Nvidia Tesla T4 de 16 GB VRAM).

### Especificaciones del Pipeline:
- **Modelo Base:** `Qwen/Qwen2.5-1.5B-Instruct` (Licencia Apache 2.0)
- **Método:** QLoRA 4-bit (bitsandbytes) + PEFT (r=16, alpha=32) + TRL SFTTrainer
- **Pérdida:** Solo sobre la respuesta JSON del asistente (`DataCollatorForCompletionOnlyLM`)
- **Consumo VRAM:** ~3.2 GB (100% dentro de la cuota gratuita de Colab T4 de 15 GB)
- **Duración estimada:** ~18-25 minutos para 3 épocas sobre ~4,200 ejemplos

In [ ]:
# 1. Verificar GPU disponible y cuota de VRAM
!nvidia-smi

In [ ]:
# 2. Instalar dependencias necesarias
!pip install -q -U torch transformers peft trl bitsandbytes accelerate datasets scipy pydantic

In [ ]:
# 3. Clonar repositorio o subir datos sintéticos
import os
from pathlib import Path

# Si estás en Colab directo, puedes subir train.jsonl y val.jsonl o clonar el repo
os.makedirs("ml/data", exist_ok=True)
os.makedirs("ml/train/output", exist_ok=True)
print("Directorios creados. Asegúrate de tener train.jsonl y val.jsonl en ml/data/")

In [ ]:
# 4. Cargar y verificar los datos sintéticos por construcción
import json

train_file = "ml/data/train.jsonl"
val_file = "ml/data/val.jsonl"

train_samples = [json.loads(line) for line in open(train_file, 'r', encoding='utf-8')]
val_samples = [json.loads(line) for line in open(val_file, 'r', encoding='utf-8')]

print(f"Muestras de entrenamiento: {len(train_samples):,}")
print(f"Muestras de validación:    {len(val_samples):,}")
print("Ejemplo de prompt:", train_samples[0]['messages'][1]['content'])
print("Ejemplo de JSON:", train_samples[0]['messages'][2]['content'])

In [ ]:
# 5. Configurar QLoRA y Cargar Modelo Base en 4-bit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import Dataset

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Cargando modelo base en cuantización de 4 bits...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 6. Preparar Dataset con ChatML y Collator para Pérdida Solo en Respuesta
def format_chatml(sample):
    text = ""
    for m in sample["messages"]:
        text += f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
    return text

train_ds = Dataset.from_dict({"text": [format_chatml(s) for s in train_samples]})
val_ds = Dataset.from_dict({"text": [format_chatml(s) for s in val_samples]})

# Enmascarar tokens antes de '<|im_start|>assistant\n'
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)

In [ ]:
# 7. Configurar Argumentos de Entrenamiento y Ejecutar SFTTrainer
training_args = TrainingArguments(
    output_dir="ml/train/output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=2048,
    data_collator=collator,
    tokenizer=tokenizer,
    args=training_args,
)

print("Iniciando fine-tuning QLoRA...")
trainer.train()
trainer.save_model("ml/train/output/final_adapter")
tokenizer.save_pretrained("ml/train/output/final_adapter")
print("¡Entrenamiento completado y adaptador LoRA guardado!")

In [ ]:
# 8. Fusionar LoRA con el Modelo Base y Exportar
from peft import PeftModel

print("Fusionando pesos LoRA con el modelo base original...")
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
)
merged_model = PeftModel.from_pretrained(base_model_fp16, "ml/train/output/final_adapter")
merged_model = merged_model.merge_and_unload()

merged_dir = "ml/export/merged_model"
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print(f"Modelo fusionado guardado en {merged_dir}")

In [ ]:
# 9. Convertir a GGUF (q4_K_M) con llama.cpp para Ollama (P8a)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!python3 llama.cpp/convert_hf_to_gguf.py ml/export/merged_model --outfile ml/export/classforge-uml-q4_k_m.gguf --outtype q4_k_m

import os
size_mb = os.path.getsize("ml/export/classforge-uml-q4_k_m.gguf") / (1024 * 1024)
print(f"Archivo GGUF generado: ml/export/classforge-uml-q4_k_m.gguf ({size_mb:.1f} MB)")